# We build here a tool that will standardize all the bhavcopy data into a standard format that can be then analysed easily.

In [1]:
%load_ext autoreload
%autoreload 2 

import sys 
sys.path.append('/Users/manishkumar/quant')

import pandas as pd
from datetime import date
from jugaad_data.nse import bhavcopy_save
from src.standardize import MAPS, STANDARD_COLS, DATE_FORMATS, detect_format, standardize, validate

RAW = '/Users/manishkumar/quant/data/raw'

In [2]:
samples = {1: date(2008, 11, 3), 2: date(2015, 6, 15), 3: date(2024, 11, 14)}
date_cols = {1: 'TIMESTAMP', 2: 'TIMESTAMP', 3: 'TradDt'}

for fmt, d in samples.items():
    df = pd.read_csv(bhavcopy_save(d, RAW), nrows=3)
    print(fmt, len(df.columns), df[date_cols[fmt]].tolist())

1 12 ['3-NOV-2008', '3-NOV-2008', '3-NOV-2008']
2 14 ['15-JUN-2015', '15-JUN-2015', '15-JUN-2015']
3 34 ['2024-11-14', '2024-11-14', '2024-11-14']


In [3]:
for fmt, d in samples.items():
    df = pd.read_csv(bhavcopy_save(d, RAW))
    parsed = pd.to_datetime(df[date_cols[fmt]], format=DATE_FORMATS[fmt])
    print(fmt, parsed.iloc[0].date(), parsed.nunique())

1 2008-11-03 1
2 2015-06-15 1
3 2024-11-14 1


In [26]:
df3 = pd.read_csv(bhavcopy_save(samples[3], RAW), keep_default_na = False)
df3['FinInstrmTp'].value_counts()

FinInstrmTp
STK    2897
Name: count, dtype: int64

In [32]:
inds = df3['SctySrs'] == 'NA'
df3.index[inds]
df3.loc[49]

TradDt                                2024-11-14
BizDt                                 2024-11-14
Sgmt                                          CM
Src                                          NSE
FinInstrmTp                                  STK
FinInstrmId                                21225
ISIN                                INE306N08292
TckrSymb                               910TCAP28
SctySrs                                       NA
XpryDt                                          
FininstrmActlXpryDt                             
StrkPric                                        
OptnTp                                          
FinInstrmNm            UNSERENCD9.1%SRIIICIII&IV
OpnPric                                  1040.33
HghPric                                  1040.33
LwPric                                   1040.33
ClsPric                                  1040.33
LastPric                                 1040.33
PrvsClsgPric                              1050.0
UndrlygPric         

In [6]:
for fmt, d in samples.items():
    df = pd.read_csv(bhavcopy_save(d, RAW), nrows = 4)
    print(fmt, detect_format(df.columns))

1 1
2 2
3 3


In [7]:
for fmt, d in samples.items():
    out = standardize(bhavcopy_save(d, RAW))
    print(fmt, out.shape, out['fmt'].unique(), out['date'].unique())

1 (1263, 14) [1] <DatetimeArray>
['2008-11-03 00:00:00']
Length: 1, dtype: datetime64[ns]
2 (1550, 14) [2] <DatetimeArray>
['2015-06-15 00:00:00']
Length: 1, dtype: datetime64[ns]
3 (2897, 14) [3] <DatetimeArray>
['2024-11-14 00:00:00']
Length: 1, dtype: datetime64[ns]


In [8]:
out.head()

,date,symbol,series,isin,open,high,low,close,last,prev_close,volume,value,trades,fmt
0,2024-11-14,APOLLOPIPE,EQ,INE126J01016,480.15,497.45,472.90,482.95,485.0,480.95,42096,2.024763e+07,3140,3
1,2024-11-14,DMART,EQ,INE192R01011,3785.20,3858.85,3769.00,3823.85,3821.5,3759.55,305991,1.171226e+09,48805,3
2,2024-11-14,GEECEE,EQ,INE916G01016,382.85,400.50,381.20,398.50,398.0,385.55,20350,8.001750e+06,1262,3
3,2024-11-14,KAMATHOTEL,EQ,INE967C01018,195.99,202.17,191.01,194.73,194.5,195.30,22881,4.510229e+06,889,3
4,2024-11-14,MAXPOSURE,SM,INE0ECC01022,90.00,90.30,88.00,89.85,89.9,87.20,15000,1.338850e+06,15,3


In [45]:
for fmt, d in samples.items():
    df = standardize(bhavcopy_save(d, RAW))
    out = validate(df)
    print(fmt, len(out))

1 0
2 0
3 0


In [46]:
import os

problems = []
for fname in sorted(os.listdir(RAW)):
    path = os.path.join(RAW, fname)
    try:
        df = standardize(path)
        bad_rows = validate(df)
        if len(bad_rows) > 0:
            print(fname, len(bad_rows), 'soft-check failures')
    except Exception as e:
        problems.append((fname, type(e).__name__, str(e)))

print(len(problems), 'files raised errors')
problems[:5]

0 files raised errors


[]

In [47]:
bad_all = []
for fname in sorted(os.listdir(RAW)):
    df = standardize(os.path.join(RAW, fname))
    bad_all.append(validate(df))

bad = pd.concat(bad_all)
print(len(bad))

bad['close_out'] = ~((bad['low'] <= bad['close']) & (bad['close'] <= bad['high']))
bad['open_out'] = ~((bad['low'] <= bad['open']) & (bad['open'] <= bad['high']))
bad['nonpos_price'] = (bad[['open', 'high', 'low', 'close']] <= 0).any(axis=1)
bad[['close_out', 'open_out', 'nonpos_price']].sum()

0


close_out       0
open_out        0
nonpos_price    0
dtype: int64

In [48]:
cb = bad[bad['close_out']].copy()
cb['gap_pct'] = ((cb['close'] - cb['high']).clip(lower=0)
                 + (cb['low'] - cb['close']).clip(lower=0)) / cb['close']

cb[['date', 'symbol', 'series', 'open', 'high', 'low', 'close',
    'last', 'prev_close', 'volume', 'gap_pct']].sort_values('gap_pct', ascending=False).head(15)

,date,symbol,series,open,high,low,close,last,prev_close,volume,gap_pct
